# 3.11 · 不平衡数据 / Imbalanced Data

> **课程定位 / Where this fits**
> **Part 3 第 11 课**。欺诈(0.1%)、罕见病、设备故障——**最有价值的类往往最稀少**。3.10 的分层 CV 解决了"评估时不丢少数类", 这一课解决"训练时学得到少数类"。核心警告：**准确率在不平衡下是骗子**。
> The valuable class is often the rarest. Accuracy is a liar under imbalance.

> 💡 **面试相关 / Interview-relevant**
> - "99% 准确率的欺诈模型好吗" ★★★★★（陷阱题——全猜没欺诈就 99%）
> - "SMOTE 原理 + 它的泄漏陷阱" ★★★★★
> - "不平衡用什么指标" ★★★★（PR-AUC / F1 / recall）
> - "class_weight 怎么work" ★★★

---

## 学习目标 / Learning Objectives
1. 理解**准确率悖论**, 改用 precision/recall/F1/PR-AUC（接 Part 5 评估）。
2. 掌握三类武器：**重采样(SMOTE) / 类权重 / 阈值调整**。
3. **SMOTE 的致命泄漏陷阱**——必须在 CV 折内、只对 train 做。
4. 形成"先调阈值/权重, 慎用合成"的实战次序。

## 目录 / TOC
1. [准确率悖论 ⭐](#1)
2. [💳 数据：极端不平衡欺诈](#2)
3. [基线：朴素模型的假象](#3)
4. [武器 1：类权重 class_weight ⭐](#4)
5. [武器 2：重采样 (欠采样/过采样/SMOTE)](#5)
6. [⚠ SMOTE 的泄漏陷阱 ⭐](#6)
7. [武器 3：阈值调整 ⭐](#7)
8. [三招对比 + 决策次序](#8)
9. [小结](#9)


<a id="1"></a>
## 1. 准确率悖论 ⭐ / The Accuracy Paradox

欺诈率 0.5% 的数据, 一个**永远预测"非欺诈"**的傻瓜模型——**准确率 99.5%**。但它**一个欺诈都没抓到**, 完全无用。

**准确率在不平衡下毫无意义**, 因为它被多数类主导。必须换指标：

| 指标 | 定义 | 不平衡下的意义 |
|---|---|---|
| Precision | TP/(TP+FP) | 报警里有多少是真的（别狼来了）|
| **Recall** | TP/(TP+FN) | 真欺诈抓到多少（别漏网）⭐ |
| **F1** | precision/recall 调和均值 | 两者平衡 |
| **PR-AUC** | precision-recall 曲线下面积 | 不平衡的首选总览指标 ⭐ |

(完整指标体系在 Part 5.2; 这里只需知道"别看准确率")。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score, average_precision_score)
from sklearn.dummy import DummyClassifier
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 极端不平衡: 1% 欺诈 / extreme imbalance
X, y = make_classification(n_samples=10000, n_features=15, n_informative=6,
                           weights=[0.99, 0.01], flip_y=0.01, random_state=0)
print(f"类别分布: {np.bincount(y)} → 欺诈占 {y.mean():.1%}")

# 傻瓜模型: 永远猜多数类 / dummy: always predict majority
dummy = DummyClassifier(strategy="most_frequent").fit(X, y)
print(f"\n傻瓜模型(永远猜非欺诈): 准确率 = {dummy.score(X, y):.1%}")
print(f"  但 recall(欺诈) = {recall_score(y, dummy.predict(X), zero_division=0):.1%} — 一个都没抓到!")
print("  → 99% 准确率 + 0% 召回 = 完全无用. 这就是准确率悖论")


<a id="2"></a>
## 2-3. 数据 + 基线的假象 / Baseline's Illusion


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

# 朴素逻辑回归 (不处理不平衡) / vanilla logistic regression
clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print("朴素逻辑回归:")
print(f"  准确率 = {clf.score(X_te, y_te):.1%}  (看着不错)")
print(f"  precision = {precision_score(y_te, y_pred):.1%}")
print(f"  recall    = {recall_score(y_te, y_pred):.1%}  ← 召回低! 漏掉大量欺诈")
print(f"  F1        = {f1_score(y_te, y_pred):.1%}")
print(f"  PR-AUC    = {average_precision_score(y_te, clf.predict_proba(X_te)[:,1]):.3f}")
print("\n混淆矩阵 (行=真实, 列=预测):")
print(confusion_matrix(y_te, y_pred))
print("→ 模型偏向多数类, 召回不足. 需要专门处理")


<a id="4"></a>
## 4. 武器 1：类权重 class_weight ⭐ / Class Weights

**最简单优雅**——不动数据, 改**损失函数**：给少数类的错误**更大惩罚**。

$$\text{加权损失} = \sum_i w_{y_i} \cdot \ell(\hat{y}_i, y_i), \qquad w_{\text{少数类}} \gg w_{\text{多数类}}$$

`class_weight="balanced"` 自动设 $w_c = \frac{n}{K \cdot n_c}$（类越小权重越大）。**线性模型/树/SVM 都支持, 零额外数据, 首选尝试**。
The simplest fix: reweight the loss so minority errors hurt more. No data manipulation, supported everywhere.


In [ ]:
# class_weight='balanced' / weighted loss
clf_w = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr)
y_pred_w = clf_w.predict(X_te)

print("逻辑回归 + class_weight='balanced':")
print(f"  准确率 = {clf_w.score(X_te, y_te):.1%}  (略降 — 正常)")
print(f"  recall    = {recall_score(y_te, y_pred_w):.1%}  ← 大幅提升! 抓到更多欺诈")
print(f"  precision = {precision_score(y_te, y_pred_w):.1%}  (降了 — 误报变多, 权衡)")
print(f"  F1        = {f1_score(y_te, y_pred_w):.1%}")
print(f"\n自动权重: 非欺诈={len(y_tr)/(2*np.bincount(y_tr)[0]):.2f}, 欺诈={len(y_tr)/(2*np.bincount(y_tr)[1]):.1f}")
print("→ 召回从基线大涨, 代价是 precision 下降 (更多误报). 这是不平衡处理的本质权衡")


<a id="5"></a>
## 5. 武器 2：重采样 / Resampling

直接改数据的类别比例。三种：

| 方法 | 做法 | 风险 |
|---|---|---|
| **随机欠采样** | 删多数类样本 | 丢信息 |
| **随机过采样** | 复制少数类 | 过拟合（重复样本）|
| **SMOTE** ⭐ | **合成**新的少数类样本（在少数类近邻间插值）| 可能造出不真实的点 |

**SMOTE 思想**：对每个少数类样本, 找它的 k 个少数类近邻, 在连线上随机取点作为新合成样本——**不是复制, 是插值**, 缓解过拟合。
SMOTE interpolates between minority-class neighbors to synthesize new points — not copying, so less overfitting.


In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

print(f"原始 train: {np.bincount(y_tr)}")
for name, sampler in [("随机欠采样", RandomUnderSampler(random_state=0)),
                      ("随机过采样", RandomOverSampler(random_state=0)),
                      ("SMOTE", SMOTE(random_state=0))]:
    Xr, yr = sampler.fit_resample(X_tr, y_tr)
    print(f"{name}: {np.bincount(yr)}")

# 用 SMOTE 后训练 / train on SMOTE-balanced data
X_sm, y_sm = SMOTE(random_state=0).fit_resample(X_tr, y_tr)
clf_sm = LogisticRegression(max_iter=1000).fit(X_sm, y_sm)
print(f"\nSMOTE + 逻辑回归: recall={recall_score(y_te, clf_sm.predict(X_te)):.1%}, "
      f"F1={f1_score(y_te, clf_sm.predict(X_te)):.1%}")


<a id="6"></a>
## 6. ⚠ SMOTE 的泄漏陷阱 ⭐ / The SMOTE Leakage Trap

**面试高频, 也是真实事故重灾区**。SMOTE 必须满足两条：
1. **只对 train 做**（绝不对 test 合成——test 必须是真实分布）
2. **在 CV 折内做**（先 SMOTE 再 CV = 合成样本泄漏到验证折）

❌ **错误**：先对全数据 SMOTE, 再 CV → 验证折里有"用 train 的点合成的样本" = 泄漏。


In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline   # imblearn 的 Pipeline (会在 fit 时才 SMOTE)

# ❌ 错: 先 SMOTE 全 train 再 CV / WRONG: SMOTE before CV
X_sm_all, y_sm_all = SMOTE(random_state=0).fit_resample(X_tr, y_tr)
f1_wrong = cross_val_score(LogisticRegression(max_iter=1000), X_sm_all, y_sm_all,
                           cv=StratifiedKFold(5), scoring="f1").mean()

# ✅ 对: SMOTE 放进 imblearn pipeline, 每折只在 train 折合成 / RIGHT: SMOTE inside pipeline
pipe = ImbPipeline([("smote", SMOTE(random_state=0)),
                    ("clf", LogisticRegression(max_iter=1000))])
f1_right = cross_val_score(pipe, X_tr, y_tr, cv=StratifiedKFold(5), scoring="f1").mean()

print(f"❌ 先 SMOTE 再 CV:        F1 = {f1_wrong:.3f}  ← 虚高 (合成样本泄漏进验证折)")
print(f"✅ SMOTE 在 pipeline 内:  F1 = {f1_right:.3f}  ← 诚实")
print(f"\n差异 = SMOTE 泄漏的乐观偏差")
print("关键: 用 imblearn.pipeline.Pipeline (不是 sklearn 的), 它只在每折的 train 上 SMOTE")
print("test 折永远是真实样本, 从不被合成污染 — 这才是诚实评估")


**为什么泄漏**：若先对全 train SMOTE 再 5-fold, 某个合成样本是用点 A 和点 B 插值的; 如果 A 进了训练折、合成样本进了验证折——模型间接见过验证折的信息。**必须用 `imblearn.pipeline.Pipeline`**, 它保证 SMOTE 只发生在每折的训练部分。
The fix: imblearn's Pipeline applies SMOTE only to each fold's training portion, never the validation fold.


<a id="7"></a>
## 7. 武器 3：阈值调整 ⭐ / Threshold Tuning

**最被低估、往往最有效**。分类器默认阈值 0.5——但**不平衡下这个默认是任意的**。降低阈值 → 更多样本判为正类 → 召回↑precision↓。

**关键洞察**：模型输出的是**概率**, 0.5 只是一个**部署时可调**的旋钮。**很多"不平衡问题"其实是"阈值没调"问题**——模型本身排序能力（AUC）是好的, 只是阈值卡错了。
The 0.5 threshold is arbitrary. Many "imbalance problems" are really "untuned threshold" problems — the model ranks fine, the cutoff is just wrong.


In [ ]:
from sklearn.metrics import precision_recall_curve

# 用基线模型的概率, 扫不同阈值 / sweep thresholds on the vanilla model's probabilities
probs = clf.predict_proba(X_te)[:, 1]
prec, rec, thresh = precision_recall_curve(y_te, probs)

# 找 F1 最大的阈值 / threshold maximizing F1
f1s = 2*prec*rec / (prec+rec+1e-9)
best_idx = np.argmax(f1s)
best_thresh = thresh[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(thresh, prec[:-1], label="precision")
axes[0].plot(thresh, rec[:-1], label="recall")
axes[0].plot(thresh, f1s[:-1], label="F1", lw=2)
axes[0].axvline(0.5, color="gray", ls="--", label="默认 0.5")
axes[0].axvline(best_thresh, color="red", ls="--", label=f"最优 {best_thresh:.2f}")
axes[0].set_xlabel("threshold"); axes[0].legend(); axes[0].set_title("阈值 vs 指标")
axes[1].plot(rec, prec); axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_title(f"PR 曲线 (AUC={average_precision_score(y_te, probs):.3f})")
plt.tight_layout(); plt.show()

print(f"默认阈值 0.5:  F1 = {f1_score(y_te, probs>0.5):.3f}")
print(f"最优阈值 {best_thresh:.2f}: F1 = {f1_score(y_te, probs>best_thresh):.3f}")
print("仅调阈值 (没碰数据/没改模型) 就提升了 F1 — 而且零成本零泄漏风险")


**阈值怎么定**：不该追求"F1 最大", 而该按**业务成本**——漏报一个欺诈损失 \$1000, 误报一次人工复核成本 \$10 → 阈值应大幅偏向高召回。**这是把统计决策（2.10 期望损失）落到部署**。
Set the threshold by business cost, not F1 — if a missed fraud costs \$1000 and a false alarm \$10, lean hard toward recall.


<a id="8"></a>
## 8. 三招对比 + 决策次序 / Comparison & Order

```
推荐尝试次序 (从低成本到高):
  1. 先换指标 — 别看准确率, 看 PR-AUC/F1/recall (零成本)
  2. 调阈值 — 模型概率排序好就够, 只调旋钮 (零成本, 无泄漏) ⭐
  3. class_weight='balanced' — 改损失, 不动数据 (一个参数)
  4. SMOTE/重采样 — 改数据 (必须 pipeline 内防泄漏) ⚠
  5. 换模型 — 树/GBDT 天然较抗不平衡; 或专门的异常检测 (Part 5.14)
```


In [ ]:
# 横向对比四种策略 (用 PR-AUC, 不平衡的可靠指标) / compare via PR-AUC
from imblearn.pipeline import Pipeline as ImbPipeline

strategies = {
    "基线":            LogisticRegression(max_iter=1000),
    "class_weight":    LogisticRegression(max_iter=1000, class_weight="balanced"),
    "SMOTE(pipeline)": ImbPipeline([("s", SMOTE(random_state=0)), ("c", LogisticRegression(max_iter=1000))]),
}
print(f"{'策略':<18} {'PR-AUC':>8} {'F1':>7} {'recall':>8}")
for name, model in strategies.items():
    pr = cross_val_score(model, X_tr, y_tr, cv=StratifiedKFold(5), scoring="average_precision").mean()
    f1 = cross_val_score(model, X_tr, y_tr, cv=StratifiedKFold(5), scoring="f1").mean()
    rc = cross_val_score(model, X_tr, y_tr, cv=StratifiedKFold(5), scoring="recall").mean()
    print(f"{name:<20} {pr:>8.3f} {f1:>7.3f} {rc:>8.3f}")
print("\n注意: PR-AUC 不太受策略影响 (它衡量排序能力, 与阈值无关)")
print("→ 重采样/权重主要改变默认阈值下的 P/R 平衡, 不一定提升排序本身")
print("→ 这就是为什么'调阈值'常常和'重采样'效果相当, 但成本低得多")


**深刻洞察**：PR-AUC 衡量的是模型的**排序能力**（与阈值无关）, 而 class_weight/SMOTE 主要改变**默认阈值下的 precision/recall 平衡**。**如果模型排序已经不错, 调阈值 = 重采样的效果但零成本**——这是很多资深 DS 的经验：先调阈值, 不行再考虑重采样。
PR-AUC measures ranking (threshold-free); reweighting mainly shifts the P/R balance at the default cutoff. If ranking is good, threshold tuning matches resampling at zero cost.


<a id="9"></a>
## 9. 小结 / Summary

```
准确率悖论: 99% 准确率可以 0 召回 → 不平衡看 PR-AUC/F1/recall
三招武器:
  class_weight='balanced' — 改损失, 零数据成本, 首选
  重采样 SMOTE — 合成少数类 (插值非复制); ⚠ 必须 imblearn pipeline 内防泄漏
  阈值调整 — 最被低估; 模型排序好就够, 零成本零泄漏 ⭐
关键洞察: PR-AUC(排序) 与阈值无关; 重采样/权重主要改阈值下的 P/R 平衡
推荐次序: 换指标 → 调阈值 → class_weight → SMOTE → 换模型
阈值按业务成本定 (漏报vs误报代价), 不是 F1 最大
```

### 💡 面试速查
1. **准确率悖论**: 全猜多数类就 99% → 用 PR-AUC/recall
2. **SMOTE 泄漏**: 必须 CV 折内、只对 train (imblearn pipeline)
3. **SMOTE = 插值合成**, 不是复制 (缓解过拟合)
4. **阈值调整最被低估**: 0.5 是任意的, 按业务成本调
5. **PR-AUC 与阈值无关** (衡量排序); 权重/重采样改的是 P/R 平衡

### 下一节
**3.12 流水线 Pipelines**——Part 3 收官。本 Part 反复说"用 Pipeline 防泄漏", 最后一课把所有预处理串成一个**结构性防泄漏 + 可部署**的整体。
